# QIIME2 Manifest Generator
This notebook replicates the behavior of the Bash script to generate a manifest CSV for QIIME2 from paired-end FASTQ files.

In [1]:
import os
import glob
import csv
import re

In [ ]:
# Path to the folder on your external drive containing the FASTQ files
# Remember that if you are using Docker you'll need to mount /data/ to /Volumes/ROSALIND/, and replace the two.
# data_dir = "/Volumes/ROSALIND/FullData12SV5/raw"  # <--- EDIT THIS
data_dir = "/Volumes/ROSALIND/s25_test/Data"

# Name of output manifest (without extension)
#LIB = "12SV5_02082026"
LIB = "s25_test"

# Forward and reverse read suffixes
FORWARD_SUFFIX = "_R1.fastq.gz"
REVERSE_SUFFIX = "_R2.fastq.gz"

# Regex to remove only the final read indicator
EXTRACT_SAMPLE_ID = r'_R[12]\.fastq\.gz$'

In [7]:
# List all FASTQ files on the external drive
forward_reads = sorted(glob.glob(os.path.join(data_dir, f'*{FORWARD_SUFFIX}')))
reverse_reads = sorted(glob.glob(os.path.join(data_dir, f'*{REVERSE_SUFFIX}')))

print("Forward reads found:")
for f in forward_reads:
    print(f"  {f}")

print("\nReverse reads found:")
for r in reverse_reads:
    print(f"  {r}")

# Quick check for matching sample IDs
print("\nSample ID preview (based on forward reads):")
for f in forward_reads:
    sample_id = re.sub(EXTRACT_SAMPLE_ID, '', os.path.basename(f))
    print(f"  {sample_id}")

Forward reads found:
  /Volumes/ROSALIND/s25_test/Data/S_SC_R1_A-Set1_R1.fastq.gz
  /Volumes/ROSALIND/s25_test/Data/S_SC_R1_A-Set2_R1.fastq.gz
  /Volumes/ROSALIND/s25_test/Data/S_SC_R1_A-Set3_R1.fastq.gz
  /Volumes/ROSALIND/s25_test/Data/S_SC_R2_A-Set1_R1.fastq.gz
  /Volumes/ROSALIND/s25_test/Data/S_SC_R2_A-Set2_R1.fastq.gz
  /Volumes/ROSALIND/s25_test/Data/S_SC_R2_A-Set3_R1.fastq.gz
  /Volumes/ROSALIND/s25_test/Data/S_SC_R3_A-Set1_R1.fastq.gz
  /Volumes/ROSALIND/s25_test/Data/S_SC_R3_A-Set2_R1.fastq.gz
  /Volumes/ROSALIND/s25_test/Data/S_SC_R3_A-Set3_R1.fastq.gz

Reverse reads found:
  /Volumes/ROSALIND/s25_test/Data/S_SC_R1_A-Set1_R2.fastq.gz
  /Volumes/ROSALIND/s25_test/Data/S_SC_R1_A-Set2_R2.fastq.gz
  /Volumes/ROSALIND/s25_test/Data/S_SC_R1_A-Set3_R2.fastq.gz
  /Volumes/ROSALIND/s25_test/Data/S_SC_R2_A-Set1_R2.fastq.gz
  /Volumes/ROSALIND/s25_test/Data/S_SC_R2_A-Set2_R2.fastq.gz
  /Volumes/ROSALIND/s25_test/Data/S_SC_R2_A-Set3_R2.fastq.gz
  /Volumes/ROSALIND/s25_test/Data/S_SC_R3_

In [8]:
manifest_file = os.path.join(data_dir, f"{LIB}.manifest.tsv")
print(f"Building manifest: {manifest_file}")

with open(manifest_file, 'w', newline='') as tsvfile:
    writer = csv.writer(tsvfile, delimiter='\t')
    writer.writerow(['sample-id', 'forward-absolute-filepath', 'reverse-absolute-filepath'])

    for f in forward_reads:
        sample_id = re.sub(EXTRACT_SAMPLE_ID, '', os.path.basename(f))
        reverse_file = os.path.join(data_dir, f"{sample_id}{REVERSE_SUFFIX}")

        if not os.path.isfile(reverse_file):
            print(f"WARNING: Missing reverse file for {sample_id}")
            continue

        forward_path = os.path.abspath(f)
        reverse_path = os.path.abspath(reverse_file)

        writer.writerow([sample_id, forward_path, reverse_path])

print(f"Manifest created: {manifest_file}")

Building manifest: /Volumes/ROSALIND/s25_test/Data/s25_test.manifest.tsv
Manifest created: /Volumes/ROSALIND/s25_test/Data/s25_test.manifest.tsv
